In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from functools import reduce

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import statsmodels.api as sm
from scipy.stats import mannwhitneyu

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

print('Cognitive Phenotypes Analysis — NHANES 2011-2014')

In [ ]:
# 1. LOAD DATA — 2011-2012 (G) + 2013-2014 (H)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

NHANES_2011_2012_DIR = DATA_DIR / "nhanes" / "2011-2012"
NHANES_2013_2014_DIR = DATA_DIR / "nhanes" / "2013-2014"

def load_g(file): return pd.read_sas(f'{PATH_G}{file}', format='xport')
def load_h(file): return pd.read_sas(f'{PATH_H}{file}', format='xport')

# Wave G (2011-2012)
demo_g = load_g('DEMO_G.xpt')
glu_g   = load_g('GLU_G.xpt')
hdl_g   = load_g('HDL_G.xpt')
trig_g  = load_g('TRIGLY_G.xpt')
bio_g   = load_g('BIOPRO_G.xpt')

# Wave H (2013-2014)
demo_h = load_h('DEMO_H.xpt')
glu_h   = load_h('GLU_H.xpt')
ins_h   = load_h('INS_H.xpt')
hdl_h   = load_h('HDL_H.xpt')
trig_h  = load_h('TRIGLY_H.xpt')
bio_h   = load_h('BIOPRO_H.xpt')

print('Loaded: Wave G (2011-2012), Wave H (2013-2014)')

In [ ]:
# 2. BUILD COMBINED DATASET
# Wave G
df_g = demo_g[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1']].copy()
df_g = df_g.rename(columns={'RIDAGEYR': 'Age', 'RIAGENDR': 'Sex', 'RIDRETH1': 'Race'})
df_g = df_g.merge(glu_g[['SEQN', 'LBXGLU', 'LBXIN']].rename(columns={'LBXGLU': 'Glucose', 'LBXIN': 'Insulin'}), on='SEQN', how='left')
df_g = df_g.merge(hdl_g[['SEQN', 'LBDHDD']].rename(columns={'LBDHDD': 'HDL'}), on='SEQN', how='left')
df_g = df_g.merge(trig_g[['SEQN', 'LBXTR']].rename(columns={'LBXTR': 'Triglycerides'}), on='SEQN', how='left')
df_g = df_g.merge(bio_g[['SEQN', 'LBXSATSI', 'LBXSGTSI', 'LBXSUA']].rename(
    columns={'LBXSATSI': 'ALT', 'LBXSGTSI': 'GGT', 'LBXSUA': 'Uric_Acid'}), on='SEQN', how='left')
df_g['wave'] = 'G'

# Wave H
df_h = demo_h[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1']].copy()
df_h = df_h.rename(columns={'RIDAGEYR': 'Age', 'RIAGENDR': 'Sex', 'RIDRETH1': 'Race'})
df_h = df_h.merge(glu_h[['SEQN', 'LBXGLU']].rename(columns={'LBXGLU': 'Glucose'}), on='SEQN', how='left')
df_h = df_h.merge(ins_h[['SEQN', 'LBXIN']].rename(columns={'LBXIN': 'Insulin'}), on='SEQN', how='left')
df_h = df_h.merge(hdl_h[['SEQN', 'LBDHDD']].rename(columns={'LBDHDD': 'HDL'}), on='SEQN', how='left')
df_h = df_h.merge(trig_h[['SEQN', 'LBXTR']].rename(columns={'LBXTR': 'Triglycerides'}), on='SEQN', how='left')
df_h = df_h.merge(bio_h[['SEQN', 'LBXSATSI', 'LBXSGTSI', 'LBXSUA']].rename(
    columns={'LBXSATSI': 'ALT', 'LBXSGTSI': 'GGT', 'LBXSUA': 'Uric_Acid'}), on='SEQN', how='left')
df_h['wave'] = 'H'

# Combine
df = pd.concat([df_g, df_h], ignore_index=True)
print(f'Total combined: {len(df)}')

# Define feature lists for later use
CLUSTER_FEATURES = ['Glucose', 'Insulin', 'Triglycerides', 'HDL', 'ALT', 'GGT', 'Uric_Acid']
race_labels = {1: 'Mexican American', 2: 'Other Hispanic', 3: 'Non-Hispanic White', 
               4: 'Non-Hispanic Black', 5: 'Other'}
cog_vars = ['DSS', 'CERAD_total', 'CERAD_immediate', 'Animal_Fluency']

In [ ]:
# 2a. DESCRIPTIVES — Full sample for clustering
print('='*70)
print('DESCRIPTIVE STATISTICS — Full Sample (2011-2014)')
print('='*70)

print(f'\nTotal participants: {len(df)}')
print(f'Age: {df["Age"].mean():.1f} ± {df["Age"].std():.1f} [range: {df["Age"].min():.0f}-{df["Age"].max():.0f}]')

print(f'\nSex distribution:')
sex_counts = df['Sex'].value_counts()
print(f'  Male (1): {int(sex_counts.get(1.0, 0))} ({sex_counts.get(1.0, 0)/len(df)*100:.1f}%)')
print(f'  Female (2): {int(sex_counts.get(2.0, 0))} ({sex_counts.get(2.0, 0)/len(df)*100:.1f}%)')

print(f'\nRace distribution (RIDRETH1):')
race_counts = df['Race'].value_counts().sort_index()
for r, c in race_counts.items():
    label = race_labels.get(r, f'Unknown ({r})')
    print(f'  {label}: {int(c)} ({c/len(df)*100:.1f}%)')

print(f'\nWave distribution:')
wave_counts = df['wave'].value_counts()
for w, c in wave_counts.items():
    label = '2011-2012' if w == 'G' else '2013-2014'
    print(f'  {label}: {int(c)} ({c/len(df)*100:.1f}%)')

print(f'\nMetabolic markers (median [IQR]):')
for var in CLUSTER_FEATURES:
    vals = df[var].dropna()
    if len(vals) > 0:
        print(f'  {var}: {vals.median():.1f} [{vals.quantile(0.25):.1f}–{vals.quantile(0.75):.1f}]')

In [ ]:
# 3. COHORT FOR CLUSTERING — all ages, with metabolic features
df_cluster = df.dropna(subset=CLUSTER_FEATURES).copy()
print(f"Cohort with all metabolic features: {len(df_cluster)}")
print(f"Age range: {df_cluster['Age'].min():.0f} - {df_cluster['Age'].max():.0f}")

In [ ]:
# 4. PREPROCESSING — log transform + z-score
LOG_VARS = ['Insulin', 'Triglycerides']

for var in LOG_VARS:
    df_cluster[f'log_{var}'] = np.log1p(df_cluster[var])

CLUSTER_VARS = [
    'Glucose', 'log_Insulin', 'log_Triglycerides', 'HDL',
    'ALT', 'GGT', 'Uric_Acid'
]

X_cluster = df_cluster[CLUSTER_VARS].values.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print('Clustering features:', CLUSTER_VARS)
print(f'Shape: {X_scaled.shape}')

In [ ]:
# 5. K-MEANS — k = 2, 3, 4 | silhouette
results_km = {}

for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results_km[k] = {'labels': labels, 'silhouette': sil, 'km': km}
    print(f'K={k}: silhouette={sil:.3f}')

BEST_K = 2
print(f'\nSelected k = {BEST_K}')

In [ ]:
# 6. FINAL CLUSTERING
km_best = results_km[BEST_K]['km']
labels_km = results_km[BEST_K]['labels']
df_cluster['Cluster'] = labels_km

print(f'Cluster distribution (K-means, k={BEST_K}):')
print(df_cluster['Cluster'].value_counts().sort_index())

In [ ]:
# 7. CLUSTER PROFILES
PROFILE_VARS = ['Glucose', 'Insulin', 'Triglycerides', 'HDL', 'ALT', 'GGT', 'Uric_Acid']

print(f'Cluster profiles (median [IQR]), k={BEST_K}\n')
print('='*80)

for c in sorted(df_cluster['Cluster'].unique()):
    n_c = (df_cluster['Cluster'] == c).sum()
    print(f'\nCluster {c} (N={n_c}, {n_c/len(df_cluster)*100:.1f}%)')
    print('-'*80)
    for var in PROFILE_VARS:
        vals = df_cluster.loc[df_cluster['Cluster']==c, var]
        med = vals.median()
        q1 = vals.quantile(0.25)
        q3 = vals.quantile(0.75)
        overall_med = df_cluster[var].median()
        direction = '↑' if med > overall_med else '↓' if med < overall_med else '='
        print(f'  {var:<20s}: {med:>7.1f} [{q1:.1f}–{q3:.1f}]  {direction}')

print('\n↑ = above overall median, ↓ = below overall median')

In [ ]:
# 7a. STATISTICAL TESTS — Mann-Whitney U test between clusters
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

print('='*80)
print('STATISTICAL COMPARISON BETWEEN CLUSTERS')
print('='*80)
print('\nMann-Whitney U test (two-sided):\n')

PROFILE_VARS = ['Glucose', 'Insulin', 'Triglycerides', 'HDL', 'ALT', 'GGT', 'Uric_Acid']

results_stats = []
for var in PROFILE_VARS:
    group0 = df_cluster[df_cluster['Cluster'] == 0][var].dropna()
    group1 = df_cluster[df_cluster['Cluster'] == 1][var].dropna()

    if len(group0) > 0 and len(group1) > 0:
        stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        med0 = group0.median()
        med1 = group1.median()
        results_stats.append({
            'Variable': var,
            'Cluster_0_median': med0,
            'Cluster_1_median': med1,
            'Difference': med1 - med0,
            'U_statistic': stat,
            'p_value': p
        })
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f'{var:20s}: Cluster 0 = {med0:7.1f}, Cluster 1 = {med1:7.1f}, Δ = {med1-med0:+6.1f}, p = {p:.4f} {sig}')


# FDR correction
results_df_stats = pd.DataFrame(results_stats)
_, p_adj, _, _ = multipletests(results_df_stats['p_value'], method='fdr_bh')
results_df_stats['p_adj'] = p_adj

print('\n' + '-'*80)
print('With FDR correction (Benjamini-Hochberg):\n')
for _, row in results_df_stats.iterrows():
    sig = '***' if row['p_adj'] < 0.001 else '**' if row['p_adj'] < 0.01 else '*' if row['p_adj'] < 0.05 else ''
    print(f"{row['Variable']:20s}: p_adj = {row['p_adj']:.4f} {sig}")

print('\nSignificance: * p<0.05, ** p<0.01, *** p<0.001')

In [ ]:
# 8. VISUALIZATION — Cluster boxplots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
palette = sns.color_palette('Set2', BEST_K)

for i, var in enumerate(PROFILE_VARS):
    ax = axes[i]
    sns.boxplot(data=df_cluster, x='Cluster', y=var, palette=palette, ax=ax)
    ax.set_title(var, fontsize=12)
    ax.set_xlabel('')
    if i % 4 != 0:
        ax.set_ylabel('')

axes[-1].axis('off')
plt.suptitle(f'Cluster Profiles (K-means, k={BEST_K})\nNHANES 2011-2014', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/cluster_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9. HEATMAP — median z-scores by cluster
cluster_medians = df_cluster.groupby('Cluster')[CLUSTER_VARS].median()
overall_medians = df_cluster[CLUSTER_VARS].median()
cluster_z = (cluster_medians - overall_medians) / df_cluster[CLUSTER_VARS].std()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cluster_z.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Median Z-score'}, ax=ax)
ax.set_title(f'Cluster Median Z-scores (k={BEST_K})\nNHANES 2011-2014', fontsize=13)
ax.set_xlabel('Cluster'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/cluster_heatmap.png', dpi=150)
plt.show()

In [ ]:
# 10. PCA VISUALIZATION
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}')
print(f'Total explained: {pca.explained_variance_ratio_.sum():.3f}')

fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette('Set2', BEST_K)
for c in sorted(df_cluster['Cluster'].unique()):
    mask = df_cluster['Cluster'].values == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=palette[c], label=f'Cluster {c}', alpha=0.5, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'PCA: Cluster Visualization (k={BEST_K})\nNHANES 2011-2014')
ax.legend()
plt.tight_layout()
plt.savefig('/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/pca_clusters.png', dpi=150)
plt.show()

In [ ]:
# 10a. FEATURE IMPORTANCE — Cohen's d and PCA Loadings
from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler

PROFILE_VARS = ["Glucose", "Insulin", "Triglycerides", "HDL", "ALT", "GGT", "Uric_Acid"]

print("="*80)
print("FEATURE IMPORTANCE FOR CLUSTER SEPARATION")
print("="*80)

# 1. Cohen's d
print("\n1. Cohen's d (standardized difference between clusters):")

cohens_d = []
for var in PROFILE_VARS:
    group0 = df_cluster[df_cluster["Cluster"] == 0][var].dropna()
    group1 = df_cluster[df_cluster["Cluster"] == 1][var].dropna()
    
    # Cohen's d = (mean1 - mean2) / pooled_std
    mean0, mean1 = group0.mean(), group1.mean()
    std0, std1 = group0.std(), group1.std()
    n0, n1 = len(group0), len(group1)
    pooled_std = np.sqrt(((n0-1)*std0**2 + (n1-1)*std1**2) / (n0+n1-2))
    d = (mean1 - mean0) / pooled_std if pooled_std > 0 else 0
    
    # Absolute d for ranking
    cohens_d.append({"Variable": var, "Cohen_d": d, "abs_d": abs(d), "mean_0": mean0, "mean_1": mean1})

d_df = pd.DataFrame(cohens_d).sort_values("abs_d", ascending=False)
for _, row in d_df.iterrows():
    effect = "large" if abs(row["Cohen_d"]) >= 0.8 else "medium" if abs(row["Cohen_d"]) >= 0.5 else "small"
    direction = "(Cluster 1 > Cluster 0)" if row["Cohen_d"] > 0 else "(Cluster 1 < Cluster 0)"
    print(f"  {row['Variable']:20s}: d = {row['Cohen_d']:+.3f}  [{effect}] {direction}")

print("\nInterpretation: |d| > 0.8 = large, 0.5-0.8 = medium, 0.2-0.5 = small")

# 2. PCA Loadings
print("\n" + "-"*80)
print("2. PCA Loadings (contribution to principal components):")

# Recalculate PCA on clustering features
X_for_pca = df_cluster[CLUSTER_VARS].values
scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_for_pca)

pca = PCA()
pca.fit(X_scaled_pca)

loadings = pd.DataFrame(pca.components_[:3].T, index=CLUSTER_VARS, columns=["PC1", "PC2", "PC3"])
print("\nLoadings for first 3 components:")
print(loadings.round(3).to_string())

print(f"\nExplained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}, PC3={pca.explained_variance_ratio_[2]:.1%}")

# Top contributors to PC1 (main cluster separation axis)
print("\nTop contributors to PC1 (main separation axis):")
pc1_sorted = loadings["PC1"].abs().sort_values(ascending=False)
for var in pc1_sorted.index:
    val = loadings.loc[var, "PC1"]
    print(f"  {var:20s}: {val:+.3f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cohen's d bar chart
ax = axes[0]
colors = ["#e74c3c" if x < 0 else "#2ecc71" for x in d_df["Cohen_d"]]
ax.barh(d_df["Variable"], d_df["Cohen_d"], color=colors)
ax.axvline(0, color="gray", lw=0.5)
ax.axvline(0.8, color="gray", ls="--", lw=0.5, alpha=0.5)
ax.axvline(-0.8, color="gray", ls="--", lw=0.5, alpha=0.5)
ax.set_xlabel("Cohen's d")
ax.set_title("Cohen's d: Cluster Separation\n(Red = Cluster 1 lower, Green = Cluster 1 higher)")

# PCA loadings bar chart for PC1
ax = axes[1]
pc1_vals = loadings["PC1"].sort_values()
colors = ["#e74c3c" if x < 0 else "#2ecc71" for x in pc1_vals]
ax.barh(pc1_vals.index, pc1_vals.values, color=colors)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("PC1 Loading")
ax.set_title(f"PCA Loadings: PC1\n(PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance)")

plt.tight_layout()
plt.savefig("/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/feature_importance.png", dpi=150)
plt.show()

In [ ]:
# 11. MERGE COGNITIVE DATA
cfq_g = load_g('CFQ_G.xpt')
cfq_h = load_h('CFQ_H.xpt')

for df in [cfq_g, cfq_h]:
    df.rename(columns={'CFDDS': 'DSS', 'CFDCSR': 'CERAD_immediate',
                     'CFDAST': 'Animal_Fluency', 'CFDCCS': 'Language'}, inplace=True)
    df['CERAD_total'] = df['CFDCST1'] + df['CFDCST2'] + df['CFDCST3']

cfq_g['wave'] = 'G'
cfq_h['wave'] = 'H'
cfq = pd.concat([cfq_g, cfq_h], ignore_index=True)
cfq = cfq[cfq['CFASTAT'] == 1]
print(f'Cognitive examined (CFASTAT=1): {len(cfq)}')

# Define cognitive variables for later use
cog_vars = ['DSS', 'CERAD_total', 'CERAD_immediate', 'Animal_Fluency']

In [ ]:
# 12. MERGE CLUSTER LABELS WITH COGNITIVE DATA (ИСПРАВЛЕННАЯ ВЕРСИЯ)

# Получаем кластерные метки
df_cluster_labels = df_cluster[['SEQN', 'Cluster']].copy()

# Загружаем демографические данные
demo_g = load_g('DEMO_G.xpt')
demo_h = load_h('DEMO_H.xpt')
demo = pd.concat([demo_g, demo_h], ignore_index=True)
demo = demo[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1']].rename(
    columns={'RIDAGEYR': 'Age', 'RIAGENDR': 'Sex', 'RIDRETH1': 'Race'}
)

# Объединяем
df_cog = cfq[['SEQN', 'DSS', 'CERAD_total', 'CERAD_immediate', 'Animal_Fluency']].merge(
    df_cluster_labels, on='SEQN', how='inner')

# Добавляем демографию
df_cog = df_cog.merge(demo, on='SEQN', how='left')

# Фильтруем возраст
df_cog = df_cog[df_cog['Age'] >= 60].copy()

# Преобразуем пол в числовой (если нужно)
df_cog['Sex'] = (df_cog['Sex'] == 1).astype(int)

print(f'Cognitive cohort (Age >= 60): {len(df_cog)}')
print(f'Cluster distribution: {df_cog["Cluster"].value_counts().to_dict()}')

In [ ]:
# 13. DESCRIPTIVES
print(f'N = {len(df_cog)}')
print(f'Age: {df_cog["Age"].mean():.1f} ± {df_cog["Age"].std():.1f}')
print(f'Sex: M = {int(df_cog["Sex"].sum())}, F = {int((df_cog["Sex"]==0).sum())}')
print(f'Race distribution:')
print(df_cog['Race'].value_counts())

In [ ]:
# 14. LINEAR REGRESSION: Cognitive ~ Phenotype + Age + Sex + Race
cog_vars = ['DSS', 'CERAD_total', 'CERAD_immediate', 'Animal_Fluency']
df_reg = df_cog.dropna(subset=cog_vars + ['Cluster', 'Age', 'Sex']).copy()
df_reg['Phenotype'] = df_reg['Cluster']

print(f'N for regression: {len(df_reg)}')
print(f'Phenotype distribution: {df_reg["Phenotype"].value_counts().to_dict()}\n')

In [ ]:
# 15. FIT REGRESSION MODELS
results = []

# Ensure numeric types
df_reg['Phenotype'] = df_reg['Phenotype'].astype(float)
df_reg['Age'] = df_reg['Age'].astype(float)
df_reg['Sex'] = df_reg['Sex'].astype(float)

for cog in cog_vars:
    X = df_reg[['Phenotype', 'Age', 'Sex']].copy()
    X = sm.add_constant(X)
    y = df_reg[cog].astype(float)
    
    model = sm.OLS(y, X).fit()
    
    results.append({
        'Test': cog, 'N': len(df_reg),
        'Phenotype_beta': model.params['Phenotype'],
        'Phenotype_se': model.bse['Phenotype'],
        'Phenotype_p': model.pvalues['Phenotype'],
        'Age_beta': model.params['Age'],
        'Sex_beta': model.params['Sex'],
        'R2': model.rsquared
    })
    
    print(f'{cog}:')
    print(f'  Phenotype: β = {model.params["Phenotype"]:.3f}, p = {model.pvalues["Phenotype"]:.4f}')
    print(f'  Age:       β = {model.params["Age"]:.3f}, p = {model.pvalues["Age"]:.4f}')
    print(f'  Sex:       β = {model.params["Sex"]:.3f}, p = {model.pvalues["Sex"]:.4f}')
    print(f'  R² = {model.rsquared:.3f}\n')

results_df = pd.DataFrame(results)

In [ ]:
# 16. MANN-WHITNEY TEST
print('='*60)
print('Mann-Whitney Test: Cognitive Scores by Phenotype')
print('='*60)

for cog in cog_vars:
    group0 = df_reg[df_reg['Phenotype'] == 0][cog].dropna()
    group1 = df_reg[df_reg['Phenotype'] == 1][cog].dropna()
    
    if len(group0) > 0 and len(group1) > 0:
        stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        print(f'\n{cog}:')
        print(f'  Cluster 0: median = {group0.median():.1f}, n={len(group0)}')
        print(f'  Cluster 1: median = {group1.median():.1f}, n={len(group1)}')
        print(f'  U-test: p = {p:.4f}')

In [ ]:
# 17. VISUALIZATION — Cognitive boxplots by Phenotype
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, cog in zip(axes, cog_vars):
    df_plot = df_reg[['Phenotype', cog]].dropna().copy()
    df_plot['Phenotype'] = df_plot['Phenotype'].map({0: 'Cluster 0', 1: 'Cluster 1'})
    colors = ['#4CAF50', '#F44336']
    bp = df_plot.boxplot(column=cog, by='Phenotype', ax=ax, patch_artist=True, return_type='dict')
    for patch, color in zip(bp[cog]['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.6)
    ax.set_title(cog)
    ax.set_xlabel(''); ax.set_ylabel('Score')

plt.suptitle(f'Cognitive Scores by Metabolic Phenotype\n(NHANES 2011-2014, Age ≥ 60, N={len(df_reg)})',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/cognitive_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 16. PARTIAL CORRELATIONS — Metabolic Markers vs Cognitive Tests

import pingouin as pi
from statsmodels.stats.multitest import multipletests

cog_vars = ["DSS", "CERAD_total", "CERAD_immediate", "Animal_Fluency"]
met_vars = ["Glucose", "Insulin", "Triglycerides", "HDL", "ALT", "GGT", "Uric_Acid"]

print("="*80)
print("PARTIAL CORRELATIONS: Metabolic Markers vs Cognitive Tests")
print("Controlling for: Age, Sex, Race")
print("="*80)

# Создаем датафрейм для корреляций, объединяя когнитивные и метаболические данные
df_corr = df_cog[['SEQN'] + cog_vars + ['Age', 'Sex', 'Race']].copy()
df_corr = df_corr.merge(
    df_cluster[['SEQN'] + met_vars], 
    on='SEQN', 
    how='inner'
)

# Фильтруем возраст >= 60
df_corr = df_corr[df_corr['Age'] >= 60].copy()

print(f"Sample size for correlations: {len(df_corr)}")

corr_matrix = pd.DataFrame(index=cog_vars, columns=met_vars, dtype=float)
pval_matrix = pd.DataFrame(index=cog_vars, columns=met_vars, dtype=float)

for cog in cog_vars:
    for met in met_vars:
        data = df_corr[[cog, met, "Age", "Sex", "Race"]].dropna()
        if len(data) > 10:
            try:
                res = pi.partial_corr(data=data, x=cog, y=met, covar=["Age", "Sex", "Race"], method="spearman")
                corr_matrix.loc[cog, met] = res["r"].iloc[0]
                pval_matrix.loc[cog, met] = res["p_val"].iloc[0]
            except Exception as e:
                print(f"Warning: Could not compute correlation for {cog} ~ {met}: {e}")
                corr_matrix.loc[cog, met] = 0
                pval_matrix.loc[cog, met] = 1

print("\nPartial correlations (Spearman, controlling for Age, Sex, Race):")
print(corr_matrix.round(3).to_string())

# FDR correction
# Получаем все p-values в виде списка
pvals_list = []
positions = [] 
for i, cog in enumerate(cog_vars):
    for j, met in enumerate(met_vars):
        p = pval_matrix.loc[cog, met]
        if not np.isnan(p):
            pvals_list.append(p)
            positions.append((i, j))

# Применяем FDR correction
_, p_adj_list, _, _ = multipletests(pvals_list, method="fdr_bh")

# Создаем матрицу скорректированных p-values
p_adj_matrix = pd.DataFrame(index=cog_vars, columns=met_vars, dtype=float)
p_adj_matrix[:] = np.nan  # заполняем NaN

for (i, j), p_adj in zip(positions, p_adj_list):
    p_adj_matrix.iloc[i, j] = p_adj

print("\n" + "-"*80)
print("Significant partial correlations (FDR-corrected, p_adj < 0.05):")
print("-"*80)

significant_pairs = []
for cog in cog_vars:
    for met in met_vars:
        p = p_adj_matrix.loc[cog, met]
        r = corr_matrix.loc[cog, met]
        if not np.isnan(p) and p < 0.05:
            print(f"  {cog} ~ {met}: r = {r:+.3f}, p_adj = {p:.4f}")
            significant_pairs.append((cog, met, r, p))

if len(significant_pairs) == 0:
    print("  No significant correlations after FDR correction.")

print("\nSignificance level: p_adj < 0.05 (Benjamini-Hochberg FDR)")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))

# Создаем тепловую карту
sns.heatmap(corr_matrix.astype(float), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "Spearman r"}, ax=ax, square=True)
        
ax.set_title("Partial Correlations: Metabolic Markers vs Cognitive Tests\n(controlling for Age, Sex, Race)")
ax.set_xlabel("Metabolic Marker")
ax.set_ylabel("Cognitive Test")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("/home/anastasia/Documents/1.Disser/Definitely_disser/NHANES/correlation_heatmap.png", dpi=150)
plt.show()

# Summary of significant findings
print("\n" + "="*80)
print("SUMMARY OF SIGNIFICANT FINDINGS")
print("="*80)
if significant_pairs:
    for cog, met, r, p in significant_pairs:
        direction = "positive" if r > 0 else "negative"
        print(f"  {met} is {direction}ly associated with {cog} (r = {r:+.3f}, p_adj = {p:.4f})")
else:
    print("  No significant partial correlations found after FDR correction.")